In [1]:
import ollama
import os
from pathlib import Path
from IPython.display import Markdown, display

In [2]:
# 链接 llama4:scout
client = ollama.Client(host='http://127.0.0.1:11434')

In [3]:
# ===== 参数 =====
QUESTIONS_DIR = Path("./Questions")
MODEL_NAME = "llama4:scout"

k = 3
temperature_list = [0.0, 0.2, 0.5, 0.8]

# 同目录下的 system prompt 文件
prompt_path = Path("./prompt1.txt")
system_prompt = prompt_path.read_text(encoding="utf-8")


def list_question_dirs(questions_dir: Path):
    return sorted(
        [p for p in questions_dir.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda x: x.name
    )


dirs = list_question_dirs(QUESTIONS_DIR)

In [4]:
display(dirs)

[PosixPath('Questions/Easy B3666'),
 PosixPath('Questions/Easy P15288'),
 PosixPath('Questions/Easy P15457'),
 PosixPath('Questions/Easy P4306'),
 PosixPath('Questions/Easy P7714'),
 PosixPath('Questions/Hard P11658'),
 PosixPath('Questions/Hard P11823'),
 PosixPath('Questions/Hard P13901'),
 PosixPath('Questions/Hard P15082'),
 PosixPath('Questions/Hard P6845'),
 PosixPath('Questions/Mid P1407'),
 PosixPath('Questions/Mid P14989'),
 PosixPath('Questions/Mid P3007'),
 PosixPath('Questions/Mid P3167'),
 PosixPath('Questions/Mid P4092')]

In [ ]:
# 对所有题目遍历输出解释
for d in dirs:
    question_path = d / "Question.txt"
    explain_dir = d / "LLM Explains"
    explain_dir.mkdir(parents=True, exist_ok=True)

    if not question_path.exists():
        print(f"skip: {question_path} not found")
        continue

    user_input = question_path.read_text(encoding="utf-8")

    global_idx = 1

    for temperature in temperature_list:
        for _ in range(k):
            try:
                response = ollama.chat(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_input},
                    ],
                    options={
                        "temperature": temperature,
                    },
                )

                output_text = response["message"]["content"]

                # 文件名：全局编号 + temperature
                output_path = explain_dir / f"{global_idx} {temperature}.txt"
                output_path.write_text(output_text, encoding="utf-8")

                print(f"done: {d.name} -> {output_path.name}")

                global_idx += 1

            except Exception as e:
                print(f"error: {d.name}, temp={temperature}, idx={global_idx} -> {e}")

In [6]:
# 测试index0题目输出解释
d = dirs[0]

question_path = d / "Question.txt"
explain_dir = d / "LLM Explains"
explain_dir.mkdir(parents=True, exist_ok=True)

if not question_path.exists():
    print(f"skip: {question_path} not found")
else:
    user_input = question_path.read_text(encoding="utf-8")

    global_idx = 1

    for temperature in temperature_list:
        for _ in range(k):
            try:
                response = ollama.chat(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_input},
                    ],
                    options={
                        "temperature": temperature,
                    },
                )

                output_text = response["message"]["content"]

                output_path = explain_dir / f"{global_idx} {temperature}.txt"
                output_path.write_text(output_text, encoding="utf-8")

                print(f"done: {d.name} -> {output_path.name}")

                global_idx += 1

            except Exception as e:
                print(f"error: {d.name}, temp={temperature}, idx={global_idx} -> {e}")

done: Easy B3666 -> 1 0.0.txt
done: Easy B3666 -> 2 0.0.txt
done: Easy B3666 -> 3 0.0.txt
done: Easy B3666 -> 4 0.2.txt
done: Easy B3666 -> 5 0.2.txt
done: Easy B3666 -> 6 0.2.txt
done: Easy B3666 -> 7 0.5.txt
done: Easy B3666 -> 8 0.5.txt
done: Easy B3666 -> 9 0.5.txt
done: Easy B3666 -> 10 0.8.txt
done: Easy B3666 -> 11 0.8.txt
done: Easy B3666 -> 12 0.8.txt
